# Train the Stock Direction Model on Google Colab or Kaggle

This notebook runs the full batch pipeline on a fresh Linux GPU runtime:

prices -> Finnhub news -> 16:00 ET alignment -> FinBERT sentiment -> features
-> chronological splits -> six models -> cost-aware backtest -> report.

**Before you start**

- Use a GPU runtime if possible (Colab T4; Kaggle T4/P100). FinBERT automatically
  uses CUDA when it is available and falls back to CPU otherwise.
- Store `FINNHUB_API_KEY` as a Colab secret or Kaggle secret. Never paste it into
  a saved cell.
- A free Finnhub key covers only about one year, so use `config.finnhub.yaml`.
  `config.yaml` keeps the specification's 2021-2025 window and needs a paid plan.
- Dependencies are installed into an isolated virtualenv, so the runtime's own
  packages are not modified.

Detailed instructions and troubleshooting: `docs/colab_kaggle.md`.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = (
    "google.colab" in sys.modules
    or "COLAB_RELEASE_TAG" in os.environ
)
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No nvidia-smi found; FinBERT will run on CPU (much slower).")


## 1. Get the project

Set `REPO_URL` to your repository. For a private repository, use a short-lived
access token in the URL or upload the project directory and set `PROJECT_DIR`
manually. Do not commit a token.


In [ ]:
REPO_URL = "https://github.com/your-org/stock-transformer.git"

PROJECT_DIR = (
    Path("/content/stock-transformer")
    if IN_COLAB
    else Path("/kaggle/working/stock-transformer")
)

pipeline_file = PROJECT_DIR / "src" / "pipeline.py"
if not pipeline_file.exists():
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise SystemExit(
            f"{PROJECT_DIR} exists but does not contain src/pipeline.py. "
            "Delete it, or set PROJECT_DIR to an existing project copy."
        )
    if "your-org" in REPO_URL:
        raise SystemExit(
            "Set REPO_URL to your repository, or upload the project and set "
            "PROJECT_DIR to its location."
        )
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
assert (Path.cwd() / "src" / "pipeline.py").exists()


## 2. Install the project environment

This creates a dedicated venv and installs `requirements-colab.txt`. The first
run downloads PyTorch and FinBERT, so it can take a few minutes. The venv is
discarded when the Colab/Kaggle session ends; rerun this cell after a restart.


In [ ]:
VENV_DIR = Path("/content/st-venv") if IN_COLAB else Path("/kaggle/working/st-venv")
PYTHON = VENV_DIR / "bin" / "python"

if not PYTHON.exists():
    print("Creating venv at", VENV_DIR)
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)

print("Installing requirements-colab.txt (this can take a few minutes)...")
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)

check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch, transformers, xgboost, pandas; "
        "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
        "'| transformers', transformers.__version__, "
        "'| xgboost', xgboost.__version__)",
    ],
    capture_output=True,
    text=True,
)
print(check.stdout.strip())
if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Dependency check failed")


## 3. Load the Finnhub key

The notebook reads `FINNHUB_API_KEY` from Colab Secrets or Kaggle Secrets. If no
secret exists, it prompts with hidden input. The key is never printed.


In [ ]:
import getpass


def load_secret(name: str):
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
            if value:
                return value
        except Exception:
            pass
    return None


FINNHUB_API_KEY = load_secret("FINNHUB_API_KEY")
if not FINNHUB_API_KEY:
    FINNHUB_API_KEY = getpass.getpass("FINNHUB_API_KEY (hidden input): ")
if not FINNHUB_API_KEY:
    raise SystemExit("No FINNHUB_API_KEY available.")

os.environ["FINNHUB_API_KEY"] = FINNHUB_API_KEY
print("Finnhub key loaded:", len(FINNHUB_API_KEY), "characters (value not printed)")

ALPHAVANTAGE_API_KEY = load_secret("ALPHAVANTAGE_API_KEY")
if ALPHAVANTAGE_API_KEY:
    os.environ["ALPHAVANTAGE_API_KEY"] = ALPHAVANTAGE_API_KEY
    print("Alpha Vantage key loaded.")


## 4. Choose the configuration

- `config.finnhub.yaml`: free-tier window (`2025-12-01..2026-11-20`), ~1 year.
- `config.yaml`: specification window (`2021-01-01..2025-06-30`), paid plan.

Keep `STAGE = "all"` for a fresh run. Set `STAGE = "train"` and then
`"backtest"` to reuse data already present in the session.


In [ ]:
CONFIG_PATH = "config.finnhub.yaml"   # free tier
# CONFIG_PATH = "config.yaml"         # paid plan, 2021-2025

STAGE = "all"   # all | prices | news | sentiment | features | train | backtest
FORCE = True
print("Config:", CONFIG_PATH, "| stage:", STAGE, "| force:", FORCE)


## 5. Run the pipeline

The pipeline writes `data/`, `artifacts/` and `reports/` under the project
directory. Expect roughly 10-20 minutes on a GPU runtime and 45-60 minutes on
CPU, depending on Finnhub rate limits and the number of headlines.


In [ ]:
cmd = [str(PYTHON), "-m", "src.pipeline", "--config", CONFIG_PATH]
if STAGE != "all":
    cmd += ["--stage", STAGE]
if FORCE:
    cmd += ["--force"]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


## 6. Inspect the results

The report states the test AUC for all six models, both baselines, the
combined-vs-single-modality comparison, the top-15 importances and the net-of-cost
backtest table.


In [ ]:
import json
from IPython.display import Image, Markdown, display

metrics_path = Path("reports/metrics.json")
if not metrics_path.exists():
    raise SystemExit("No reports/metrics.json yet; run the pipeline cell first.")

metrics = json.loads(metrics_path.read_text())
print("Config:", metrics["metadata"].get("config_path"))
print("Requested range:", metrics["metadata"].get("date_range"))
print("Dataset:", metrics.get("dataset"))

lines = [
    "| Model | Test AUC | Accuracy | Brier | N |",
    "|---|---:|---:|---:|---:|",
]
for name, payload in metrics["models"].items():
    test = payload["test"]
    lines.append(
        f"| {name} | {test['auc']:.4f} | {test['accuracy']:.4f} | "
        f"{test['brier']:.4f} | {test['n_samples']} |"
    )
display(Markdown("\n".join(lines)))

summary_path = Path("reports/summary.md")
if summary_path.exists():
    display(Markdown(summary_path.read_text()))

for plot in (
    "reports/equity_combined.png",
    "reports/feature_importance_combined_xgb.png",
):
    if Path(plot).exists():
        display(Image(filename=plot))


## 7. Download the outputs

Colab downloads the zip automatically. On Kaggle, download it from the Output
tab of the notebook.


In [ ]:
OUTPUT_ZIP = Path("stock-transformer-outputs.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

candidates = [
    "reports",
    "artifacts",
    "data/processed",
    "data/raw/prices.parquet",
    "data/raw/news.parquet",
]
existing = [path for path in candidates if Path(path).exists()]

import zipfile

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in existing:
        item = Path(path)
        if item.is_file():
            archive.write(item, item.as_posix())
        else:
            for child in item.rglob("*"):
                if child.is_file():
                    archive.write(child, child.as_posix())

print("Created", OUTPUT_ZIP, f"{OUTPUT_ZIP.stat().st_size / 1e6:.1f} MB")

if IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_ZIP))
else:
    print("Kaggle: download from the Output tab:", Path.cwd() / OUTPUT_ZIP)


## Optional: persist to Google Drive

Colab disconnects after a while. Uncomment and run this cell to copy the report
and model artifacts to Drive.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# target = Path("/content/drive/MyDrive/stock-transformer")
# target.mkdir(parents=True, exist_ok=True)
# shutil.copytree("reports", target / "reports", dirs_exist_ok=True)
# shutil.copytree("artifacts", target / "artifacts", dirs_exist_ok=True)
# print("Copied reports and artifacts to", target)


## Troubleshooting

See `docs/colab_kaggle.md` for the full checklist. Common issues:

- `Set REPO_URL ...`: edit the clone cell with your repository URL.
- `FINNHUB_API_KEY` missing or HTTP 401/403: add the secret and grant the
  notebook access to it.
- Zero headlines: a free key cannot read 2021-2025; use `config.finnhub.yaml`.
- CUDA out of memory: lower `sentiment.batch_size` (for example 16 or 32) in the
  config and rerun the sentiment stage.
- Yahoo certificate warnings are non-fatal; the download continues.
